# 004 — Vector Storage Types
**Storage Series** | ChromaDB · FAISS · In-memory comparison

Different vector stores suit different scales and use cases.
This notebook benchmarks ChromaDB (disk-persistent) vs FAISS (in-memory ANN) vs
a simple numpy dot-product search.


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile
✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)


In [3]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


✓ 5 dataset files loaded
  File                    Chars   ~Words
  ----------------------------------------
  ai.txt                  6,221      943
  machine_learning.txt    5,953      870
  nlp.txt                 5,160      693
  deep_learning.txt       5,114      686
  rag.txt                 5,146      715
  ----------------------------------------
  TOTAL                  27,602    3,907


In [4]:
# ── Prepare document corpus ──────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import time, numpy as np

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
raw_chunks = splitter.split_text(all_text[:20000])
docs = [Document(page_content=c, metadata={"source": "wikipedia", "idx": i})
        for i, c in enumerate(raw_chunks)]
print(f"✓ {len(docs)} documents prepared")


✓ 74 documents prepared


In [5]:
# ── Store 1: FAISS (in-memory ANN) ───────────────────────────────────────
from langchain_community.vectorstores import FAISS

t0 = time.time()
faiss_store = FAISS.from_documents(docs, embeddings)
t_faiss_build = time.time() - t0
print(f"FAISS build: {t_faiss_build:.2f}s  |  {len(docs)} vectors")

query = "How do neural networks learn from data?"
t0 = time.time()
faiss_results = faiss_store.similarity_search_with_score(query, k=4)
t_faiss_query = time.time() - t0
print(f"FAISS query: {t_faiss_query*1000:.1f}ms")
print("\nTop FAISS result:")
print(faiss_results[0][0].page_content[:250])
print(f"  L2 distance: {faiss_results[0][1]:.4f}")


FAISS build: 0.99s  |  74 vectors
FAISS query: 7.7ms

Top FAISS result:
through unsupervised learning. Some implementations of machine learning use data and neural
networks in a way that mimics the working of a biological brain.
  L2 distance: 0.7589


In [6]:
# ── Store 2: ChromaDB (persistent, filterable) ────────────────────────────
from langchain_chroma import Chroma
import chromadb

client = chromadb.Client()  # ephemeral for this demo

t0 = time.time()
chroma_store = Chroma.from_documents(docs, embeddings,
                                      client=client,
                                      collection_name="rag_demo")
t_chroma_build = time.time() - t0
print(f"ChromaDB build: {t_chroma_build:.2f}s  |  {len(docs)} vectors")

t0 = time.time()
chroma_results = chroma_store.similarity_search_with_score(query, k=4)
t_chroma_query = time.time() - t0
print(f"ChromaDB query: {t_chroma_query*1000:.1f}ms")
print("\nTop ChromaDB result:")
print(chroma_results[0][0].page_content[:250])
print(f"  Distance: {chroma_results[0][1]:.4f}")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


ChromaDB build: 1.02s  |  74 vectors
ChromaDB query: 11.6ms

Top ChromaDB result:
through unsupervised learning. Some implementations of machine learning use data and neural
networks in a way that mimics the working of a biological brain.
  Distance: 0.7589


In [7]:
# ── Store 3: Numpy brute-force (baseline) ────────────────────────────────
all_texts = [d.page_content for d in docs]
t0 = time.time()
all_vecs = np.array(embeddings.embed_documents(all_texts))
query_vec = np.array(embeddings.embed_query(query))
t_np_build = time.time() - t0

t0 = time.time()
sims = np.dot(all_vecs, query_vec) / (
    np.linalg.norm(all_vecs, axis=1) * np.linalg.norm(query_vec) + 1e-10
)
top_idx = np.argsort(sims)[::-1][:4]
t_np_query = time.time() - t0
print(f"Numpy build : {t_np_build:.2f}s")
print(f"Numpy query : {t_np_query*1000:.2f}ms")
print("\nTop numpy result:")
print(docs[top_idx[0]].page_content[:250])
print(f"  Cosine sim: {sims[top_idx[0]]:.4f}")


Numpy build : 1.02s
Numpy query : 0.58ms

Top numpy result:
through unsupervised learning. Some implementations of machine learning use data and neural
networks in a way that mimics the working of a biological brain.
  Cosine sim: 0.6206


In [8]:
# ── Benchmark summary ────────────────────────────────────────────────────
import pandas as pd

# Check if top results agree across stores
faiss_top  = faiss_results[0][0].page_content[:60]
chroma_top = chroma_results[0][0].page_content[:60]
np_top     = docs[top_idx[0]].page_content[:60]
agreement  = len({faiss_top, chroma_top, np_top})
print(f"Top-result agreement across stores: {4 - agreement}/3 match\n")

df = pd.DataFrame({
    "Store":        ["FAISS", "ChromaDB", "Numpy brute-force"],
    "Build (s)":    [f"{t_faiss_build:.2f}", f"{t_chroma_build:.2f}", f"{t_np_build:.2f}"],
    "Query (ms)":   [f"{t_faiss_query*1000:.1f}", f"{t_chroma_query*1000:.1f}",
                     f"{t_np_query*1000:.2f}"],
    "Persistence":  ["save_local()", "SQLite/disk", "None"],
    "Filtering":    ["Limited", "Full metadata", "Manual"],
    "Scale":        [">10M docs", "<1M docs", "<50k docs"],
    "Recommended":  ["Production", "Dev/Medium prod", "Prototyping"],
})
pd.set_option('display.max_colwidth', 22)
pd.set_option('display.width', 130)
print(df.to_string(index=False))


Top-result agreement across stores: 3/3 match

            Store Build (s) Query (ms)  Persistence     Filtering     Scale     Recommended
            FAISS      0.99        7.7 save_local()       Limited >10M docs      Production
         ChromaDB      1.02       11.6  SQLite/disk Full metadata  <1M docs Dev/Medium prod
Numpy brute-force      1.02       0.58         None        Manual <50k docs     Prototyping


## Key Takeaways
| Store | Choose when… |
|---|---|
| **FAISS** | Speed matters, scale > 100k docs, no persistence needed in-flight |
| **ChromaDB** | Dev/medium prod, rich metadata filtering, persistence matters |
| **Qdrant** | Production, distributed, full filtering + payload indexing |
| **pgvector** | Already on Postgres, want SQL + vector in one place |
